# Clasificación de animales (15 clases) con Transfer Learning — versión local

Estructura del proyecto (NNApp-main):

```
NNApp-main/
├── datasets/animal_data/Bear, Bird, Cat, ...
├── models/                 <- aquí se guardan los modelos
└── notebooks/Animal_Data_local.ipynb
```

**Cambio clave respecto a la versión de Colab:** el split train/val/test se hace por lista de archivos
(estratificado), así no hay fuga de datos ni clases faltantes en validación.

In [ ]:
import os, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))   # lista vacía = CPU (funciona igual, solo más lento)

# ---------- Rutas: se detecta la raíz del proyecto (la carpeta que contiene 'datasets') ----------
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No encontré la carpeta 'datasets' subiendo desde {Path.cwd()}")

DATA_ROOT  = PROJECT_ROOT / "datasets" / "animal_data"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

IMG_SIZE  = (224, 224)
BATCH     = 32
AUTOTUNE  = tf.data.AUTOTUNE

In [ ]:
# Si las clases están dentro de una subcarpeta extra (animal_data/animal_data/Bear), la detecta
if not any(d.name.lower() == "bear" for d in DATA_ROOT.iterdir() if d.is_dir()):
    for root, dirs, _ in os.walk(DATA_ROOT):
        if any(d.lower() == "bear" for d in dirs):
            DATA_ROOT = Path(root); break

print("Dataset en:", DATA_ROOT.resolve())
class_names = sorted(d.name for d in DATA_ROOT.iterdir() if d.is_dir())
NUM_CLASSES = len(class_names)
print(NUM_CLASSES, "clases:", class_names)

In [ ]:
# Lista de archivos válidos (se descartan imágenes corruptas)
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
paths, labels, skipped = [], [], 0
for idx, cname in enumerate(class_names):
    for f in sorted((DATA_ROOT / cname).iterdir()):
        if f.suffix.lower() not in EXTS:
            continue
        try:
            with Image.open(f) as im:
                im.verify()
            paths.append(str(f)); labels.append(idx)
        except Exception:
            skipped += 1
print(f"{len(paths)} imágenes válidas, {skipped} descartadas")

paths, labels = np.array(paths), np.array(labels)

# Split estratificado 70 / 15 / 15 (sin solapamiento)
p_train, p_tmp, y_train, y_tmp = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
p_val, p_test, y_val, y_test = train_test_split(
    p_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

assert not (set(p_train) & set(p_val) | set(p_train) & set(p_test) | set(p_val) & set(p_test))
print("train:", len(p_train), "| val:", len(p_val), "| test:", len(p_test))
print("Clases en val:", len(set(y_val)), "| clases en test:", len(set(y_test)))

In [ ]:
def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def make_ds(p, y, training):
    ds = tf.data.Dataset.from_tensor_slices((p, y))
    if training:
        ds = ds.shuffle(len(p), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_img, num_parallel_calls=AUTOTUNE).batch(BATCH)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(p_train, y_train, training=True)
val_ds   = make_ds(p_val,   y_val,   training=False)
test_ds  = make_ds(p_test,  y_test,  training=False)   # sin shuffle: el orden coincide con y_test

In [ ]:
# Vista rápida: una imagen por clase
fig, axes = plt.subplots(3, 5, figsize=(14, 8))
for ax, (i, cname) in zip(axes.ravel(), enumerate(class_names)):
    sample = random.choice(paths[labels == i])
    ax.imshow(Image.open(sample)); ax.set_title(cname); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = models.Model(inputs, outputs)
model.summary()

In [ ]:
def get_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        ModelCheckpoint(str(MODELS_DIR / "best_animal_model.keras"), monitor="val_loss", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
    ]

# Fase 1: entrenar solo la cabeza
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=get_callbacks())

In [ ]:
# Fase 2: fine-tuning de las últimas 30 capas con LR bajo
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=get_callbacks())

In [ ]:
# Curvas de entrenamiento (ambas fases)
acc  = history.history["accuracy"]     + history_ft.history["accuracy"]
vacc = history.history["val_accuracy"] + history_ft.history["val_accuracy"]
loss = history.history["loss"]         + history_ft.history["loss"]
vloss= history.history["val_loss"]     + history_ft.history["val_loss"]
n1 = len(history.history["accuracy"])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (t, v, name) in zip(ax, [(acc, vacc, "Accuracy"), (loss, vloss, "Loss")]):
    a.plot(t, label="train"); a.plot(v, label="val")
    a.axvline(n1 - 1, color="gray", ls="--", label="inicio fine-tuning")
    a.set_title(name); a.set_xlabel("época"); a.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Evaluación final sobre TEST (datos que el modelo nunca usó ni para decidir early stopping)
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)
labels_idx = list(range(NUM_CLASSES))
print(classification_report(y_test, y_pred, labels=labels_idx, target_names=class_names))

cm = confusion_matrix(y_test, y_pred, labels=labels_idx)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicción"); plt.ylabel("Real"); plt.title("Matriz de confusión (test)")
plt.show()

In [ ]:
model.save(MODELS_DIR / "modelo_animales.keras")
print("Modelo guardado en", MODELS_DIR / "modelo_animales.keras")

# Ejemplo de predicción con una imagen nueva
def predict_image(path):
    img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
    arr = tf.expand_dims(tf.keras.utils.img_to_array(img), 0)
    probs = model.predict(arr, verbose=0)[0]
    top = np.argsort(probs)[::-1][:3]
    return [(class_names[i], float(probs[i])) for i in top]

# print(predict_image("mi_foto.jpg"))